# Data Cleaning & ETL

This notebook focuses on cleaning, transforming and validating the raw supply chain dataset to create a business-ready analytical dataset for downstream SQL modeling and dashboard development.

## Cleaning Objectives

- Remove low-value and redundant columns
- Handle missing values
- Validate duplicate and overlapping fields
- Convert incorrect datatypes
- Standardize column naming conventions
- Improve overall data quality for analytics and reporting

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Dataset Loading

Loading the raw supply chain dataset into a Pandas DataFrame for cleaning and transformation.

In [3]:
df = pd.read_csv(
    "../data/raw/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

# Initial Dataset Shape

Checking the number of rows and columns before cleaning begins.

In [4]:
df.shape

(180519, 53)

# Removing Low-Value Columns

Dropping columns identified during the profiling phase as:
- structurally empty
- highly incomplete
- redundant
- low analytical value
- non-business-critical

In [6]:
drop_columns = [
    "Product Description",
    "Product Image",
    "Customer Password",
    "Order Zipcode",
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Status"
]

df = df.drop(columns=drop_columns)

# Dataset Shape After Column Removal

Validating the impact of column removal on the dataset structure.

In [7]:
df.shape

(180519, 44)

# Datatype Conversion

Converting date-related columns from string format to proper datetime format to enable:
- time-series analysis
- trend monitoring
- monthly KPI calculations
- shipment delay analysis

In [8]:
df[
    [
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ]
].dtypes

order date (DateOrders)       str
shipping date (DateOrders)    str
dtype: object

In [9]:
df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"]
)

df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"]
)

In [10]:
df[
    [
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ]
].dtypes

order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object

# Duplicate Validation

Checking the dataset for duplicate records to ensure transactional integrity and improve overall data quality before feature engineering and downstream analytics.

In [11]:
df.duplicated().sum()

np.int64(0)

# Transactional Integrity Checks

Validating key transactional identifiers to better understand dataset granularity and ensure data consistency before downstream analytics and warehouse modeling.

In [12]:
df["Order Id"].duplicated().sum()

np.int64(114767)

In [13]:
df["Order Item Id"].duplicated().sum()

np.int64(0)

# Redundancy Validation

Checking potentially overlapping columns to determine whether they contain identical information or represent different business hierarchies.

In [14]:
(
    df["Product Card Id"]
    ==
    df["Order Item Cardprod Id"]
).all()

np.True_

In [15]:
(
    df["Category Id"]
    ==
    df["Product Category Id"]
).all()

np.True_

## Redundancy Validation Findings

The following column pairs were validated and found to contain identical values across all rows:

- Product Card Id and Order Item Cardprod Id
- Category Id and Product Category Id

Since these columns provide duplicate information, the following columns will be removed during the cleaning phase:

- Order Item Cardprod Id
- Product Category Id

In [16]:
df = df.drop(
    columns=[
        "Order Item Cardprod Id",
        "Product Category Id"
    ]
)

In [17]:
df.shape

(180519, 42)

# Column Name Standardization

Standardizing column names into lowercase snake_case format to improve:
- readability
- SQL compatibility
- maintainability
- downstream analytics workflows

The transformation includes:
- converting text to lowercase
- replacing spaces with underscores
- removing special characters and brackets

In [18]:
df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Id',
 'Customer Segment',
 'Customer State',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Card Id',
 'Product Name',
 'Product Price',
 'shipping date (DateOrders)',
 'Shipping Mode']

In [19]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

In [20]:
df.columns.tolist()

['type',
 'days_for_shipping_real',
 'days_for_shipment_scheduled',
 'benefit_per_order',
 'sales_per_customer',
 'delivery_status',
 'late_delivery_risk',
 'category_id',
 'category_name',
 'customer_city',
 'customer_country',
 'customer_id',
 'customer_segment',
 'customer_state',
 'customer_zipcode',
 'department_id',
 'department_name',
 'latitude',
 'longitude',
 'market',
 'order_city',
 'order_country',
 'order_customer_id',
 'order_date_dateorders',
 'order_id',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_id',
 'order_item_product_price',
 'order_item_profit_ratio',
 'order_item_quantity',
 'sales',
 'order_item_total',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'order_status',
 'product_card_id',
 'product_name',
 'product_price',
 'shipping_date_dateorders',
 'shipping_mode']

# Remaining Missing Value Analysis

Rechecking the dataset after initial cleaning steps to identify any remaining missing values that may require treatment before feature engineering and analytics modeling.

In [21]:
remaining_nulls = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

remaining_nulls[remaining_nulls > 0]

customer_zipcode    3
dtype: int64

# Categorical Value Validation

Inspecting important categorical dimensions to identify inconsistent labels, formatting issues or unexpected category values that may affect reporting accuracy.

In [22]:
df["shipping_mode"].unique()

<StringArray>
['Standard Class', 'First Class', 'Second Class', 'Same Day']
Length: 4, dtype: str

In [23]:
df["delivery_status"].unique()

<StringArray>
['Advance shipping', 'Late delivery', 'Shipping on time', 'Shipping canceled']
Length: 4, dtype: str

In [24]:
df["order_status"].unique()

<StringArray>
[       'COMPLETE',         'PENDING',          'CLOSED', 'PENDING_PAYMENT',
        'CANCELED',      'PROCESSING', 'SUSPECTED_FRAUD',         'ON_HOLD',
  'PAYMENT_REVIEW']
Length: 9, dtype: str

In [25]:
df["customer_segment"].unique()

<StringArray>
['Consumer', 'Home Office', 'Corporate']
Length: 3, dtype: str

In [26]:
remaining_nulls = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

remaining_nulls[remaining_nulls > 0]

customer_zipcode    3
dtype: int64

## Remaining Missing Values Observation

Only the `customer_zipcode` column contains remaining missing values after cleaning.

- Missing Records: 3
- Business Impact: Negligible

Since zipcode is not a critical KPI-driving field for this project, the missing values will be retained without imputation.

In [27]:
df["shipping_mode"].unique()

<StringArray>
['Standard Class', 'First Class', 'Second Class', 'Same Day']
Length: 4, dtype: str

# Exporting Cleaned Dataset

Exporting the cleaned and standardized dataset to the processed data layer for downstream feature engineering, SQL modeling and dashboard development.

In [33]:
df.to_csv(
    "../data/processed/cleaned_supply_chain.csv",
    index=False
)